## Автоматическое определение необходимости эквализации изображения
Данные скрипты используют свойства гистограммы изображения для автоматического (без участия человека) принятия решения о необходимости эквализации изображения.

Использованные изображения: [BSDS300](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/bsds/)

Пусть критерием "выровненного" изображения будет являться (приблизительная) линейность эмпирической функции распределения (т. е. интеграла от гистограммы).
Это означает, что распределение цветов в самой гистограмме более-менее равномерно — как первая производная линейной функции, это будет функция с постоянным значением. Тогда производная от этой функции (вторая производная от эмпирической CDF) будет равна нулю (опять же, приблизительно).

Например, критерий может быть таким: производная гистограммы в основном имеет очень малое значение, без значительных отклонений от нуля.

### Определение по производной

Изображение можно считать "корректным", если, например, производная на протяжении всего диапазона будет иметь сравнительно низкие (по модулю) значения.

Возможно, также придётся "сглаживать" производную, используя, например, скользящее среднее: так как мы имеем дело с дискретными величинами, в противном случае присутствовало бы много резких скачков.

In [ ]:
from random import randint;

# папка с изображениями
FOLDER = './_stuff';
# индексы (с 0) изображений, которые будут показываться
IMAGES = [randint(0, 1023) for i in range(10)]

# пороговое отклонение (в долях от 1) значения производной от 0
DERIVATIVE_THRESHOLD = 0.05;
# размер окна для сглаживания (скользящее среднее)
WINDOW_SIZE = 7;

In [ ]:
import os;
import cv2 as cv;
import numpy as np;
import matplotlib.pyplot as plt;


def makeCdf(histogram):
    return np.cumsum(histogram);


# Получаем список файлов
files = sorted([
    os.path.join(FOLDER, name)
    for name in os.listdir(FOLDER)]);


for index in IMAGES:
    image_path = files[index % len(files)];

    img_bgr = cv.imread(image_path);

    gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY);

    # Гистограмма яркости
    histogram = cv.calcHist([gray], [0], None, [256], [0, 256]);
    n_pixels = sum(histogram);

    # CDF
    cdf = makeCdf(histogram);

    # производная гистограммы
    derivative = np.diff(histogram);
    # Вычисление скользящего среднего
    # предположим, что WINDOW_SIZE всегда нечётное
    new_der = derivative;
    if WINDOW_SIZE != 1:
        new_der = np.empty_like(derivative,
            dtype=np.float32)[WINDOW_SIZE // 2 : -WINDOW_SIZE // 2];
        sumval = sum(derivative[:WINDOW_SIZE]);
        for i in range(len(new_der)):
            new_der[i] = sumval;
            sumval = sumval - derivative[i] + derivative[i + WINDOW_SIZE];
        new_der /= WINDOW_SIZE;
    

    bins = np.arange(256);
    derivative_bins = np.arange(
        WINDOW_SIZE // 2 + 1,
        255 - WINDOW_SIZE // 2
    ) if WINDOW_SIZE != 1 else np.arange(1, 256);
    fig, ax = plt.subplots(2, 2, figsize=(14, 8));

    # Изображение
    ax[0, 0].imshow(gray, cmap='gray', vmin=0, vmax=255);
    ax[0, 0].set_title(os.path.basename(image_path));
    ax[0, 0].axis('off');

    # Гистограмма и CDF
    ax[0, 1].bar(bins, histogram, color='black', alpha=0.4);
    ax[0, 1].set_title('Гистограмма и CDF');
    ax[0, 1].set_ylabel('Количество пикселей');
    ax[0, 1].set_xlim(0, 255);

    ax_cdf = ax[0, 1].twinx();
    ax_cdf.plot(bins, cdf, color='red');
    ax_cdf.set_ylabel('CDF');

    # Гистограмма и производная
    ax[1, 1].bar(bins, histogram, color='black', alpha=0.4);
    ax[1, 1].set_title('Гистограмма и производная');
    ax[1, 1].set_xlabel('Уровень яркости');
    ax[1, 1].set_ylabel('Количество пикселей');
    ax[1, 1].set_xlim(0, 255);

    ax_derivative = ax[1, 1].twinx();
    if WINDOW_SIZE != 1:
        pass
        # derivative_bins = derivative_bins[WINDOW_SIZE // 2 : -WINDOW_SIZE // 2];
    ax_derivative.plot(np.arange(1, 256),
        derivative, color='blue', alpha=0); 
    ax_derivative.plot(derivative_bins, new_der, color='blue');
    ax_derivative.set_ylabel('Производная гистограммы');
    ax_derivative.axhline(0, color='gray', linewidth=0.7, alpha=0.4);
    ax_derivative.axhline(n_pixels / 256 * DERIVATIVE_THRESHOLD, color='red', linewidth=0.7, alpha=0.4);
    ax_derivative.axhline(n_pixels / -256 * DERIVATIVE_THRESHOLD, color='red', linewidth=0.7, alpha=0.4);
    
    
    # Информация о пороге и скачках производной
    ax[1, 0].axis('off');

    threshold = n_pixels / 256 * DERIVATIVE_THRESHOLD;

    jumps = np.abs(new_der);
    max_jump = np.max(jumps);

    in_range = max_jump <= threshold;

    info = (
        f"Порог производной: "
        f"{DERIVATIVE_THRESHOLD * 100:.2f}%"
        f" ({threshold:.2f})\n\n"
        f"Наибольший скачок производной: "
        f"{max_jump:.2f}\n\n"
        f"Наибольший скачок в пределах порога: "
        f"{'Да' if in_range else 'Нет'}"
    );

    ax[1, 0].text(
        0.05, 0.95, info,
        transform=ax[1, 0].transAxes,
        verticalalignment='top',
        fontsize=12
    );

    fig.tight_layout();
    plt.show();